#### Import Libraries

In [1]:
import pandas as pd
import logging 
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

#### Import Data

In [2]:
# Load call and put options data from CSV files
call_options_data = pd.read_csv(r"C:\Algorthmic Trading for Beginners\Data Management\datamodules\minute_options_data\call_options_data.csv")
put_options_data = pd.read_csv(r"C:\Algorthmic Trading for Beginners\Data Management\datamodules\minute_options_data\put_options_data.csv")

# Display column names of both datasets
print(call_options_data.columns)
print(put_options_data.columns)

Index(['Unnamed: 0', 'symbol', 'underlying_last', 'expire_date', 'expiry_type',
       'dte', 'strike', 'strike_distance', 'strike_distance_pct', 'c_delta',
       'c_gamma', 'c_vega', 'c_theta', 'c_rho', 'c_iv', 'c_volume', 'c_last',
       'c_size', 'c_bid', 'c_ask', 'c_mid', 'c_spread'],
      dtype='str')
Index(['Unnamed: 0', 'symbol', 'underlying_last', 'expire_date', 'expiry_type',
       'dte', 'strike', 'strike_distance', 'strike_distance_pct', 'p_delta',
       'p_gamma', 'p_vega', 'p_theta', 'p_rho', 'p_iv', 'p_volume', 'p_last',
       'p_size', 'p_bid', 'p_ask', 'p_mid', 'p_spread'],
      dtype='str')


In [3]:
# Rename the first column to 'quote_date' for both datasets
call_options_data = call_options_data.rename(columns={'Unnamed: 0': 'quote_date'})
put_options_data = put_options_data.rename(columns={'Unnamed: 0': 'quote_date'})

# Display updated column names
print(call_options_data.columns)
print(put_options_data.columns)

Index(['quote_date', 'symbol', 'underlying_last', 'expire_date', 'expiry_type',
       'dte', 'strike', 'strike_distance', 'strike_distance_pct', 'c_delta',
       'c_gamma', 'c_vega', 'c_theta', 'c_rho', 'c_iv', 'c_volume', 'c_last',
       'c_size', 'c_bid', 'c_ask', 'c_mid', 'c_spread'],
      dtype='str')
Index(['quote_date', 'symbol', 'underlying_last', 'expire_date', 'expiry_type',
       'dte', 'strike', 'strike_distance', 'strike_distance_pct', 'p_delta',
       'p_gamma', 'p_vega', 'p_theta', 'p_rho', 'p_iv', 'p_volume', 'p_last',
       'p_size', 'p_bid', 'p_ask', 'p_mid', 'p_spread'],
      dtype='str')


### 1) Design & Testing Schema

- This is the process of building out the blueprint of how you want your Database Schema to be structured and then testing it to ensure it will work as intended during deployment

#### 1.1) Set up Database Logger for Transparency

In [4]:
# Setup Logger
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)


#### 1.2) Setup SQLite Database

In [5]:
# db configs
db_path = r"C:\Algorthmic Trading for Beginners\Data Management\Database\options_data.db"
engine = create_engine(f"sqlite:///{db_path}")

In [6]:
# Enable FK constraints & PRAGMA optimization Settings
with engine.connect() as conn:
    conn.execute(text("PRAGMA foreign_keys=ON"))
    conn.execute(text("PRAGMA journal_mode=WAL"))
    conn.execute(text("PRAGMA synchronous=NORMAL"))

#### 1.3) Define SQL DB Tables

In [7]:
# Create Call options table 
create_call_table = """
CREATE TABLE IF NOT EXISTS call_options_data (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    quote_date TEXT NOT NULL,
    symbol TEXT NOT NULL,
    underlying_last REAL NOT NULL,
    expire_date TEXT NOT NULL,
    expiry_type TEXT,
    dte REAL,
    strike REAL NOT NULL,
    strike_distance REAL,
    strike_distance_pct REAL,
    c_delta REAL,
    c_gamma REAL, 
    c_vega REAL, 
    c_theta REAL,
    c_rho REAL, 
    c_iv REAL,
    c_volume REAL, 
    c_last REAL,
    c_size TEXT, 
    c_bid REAL, 
    c_ask REAL, 
    c_mid REAL, 
    c_spread REAL
);
"""

# Create Put options Table
create_put_table = """
CREATE TABLE IF NOT EXISTS put_options_data (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    quote_date TEXT NOT NULL,
    symbol TEXT NOT NULL,
    underlying_last REAL NOT NULL,
    expire_date TEXT NOT NULL,
    expiry_type TEXT,
    dte REAL,
    strike REAL NOT NULL,
    strike_distance REAL,
    strike_distance_pct REAL,
    p_delta REAL,
    p_gamma REAL, 
    p_vega REAL, 
    p_theta REAL,
    p_rho REAL, 
    p_iv REAL,
    p_volume REAL, 
    p_last REAL,
    p_size TEXT, 
    p_bid REAL, 
    p_ask REAL, 
    p_mid REAL, 
    p_spread REAL
);
"""

#### 1.4) Define Indexes for Tables

In [8]:
# We use .connect() to intialize a connection to the db
with engine.connect() as conn:
    logger.info("Creating call_options_data table...")
    conn.execute(text(create_call_table))

    logger.info("Creating put_options_data table...")
    conn.execute(text(create_put_table))

    # Add Indexes for CALL table
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_call_symbol ON call_options_data(symbol);"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_call_quote_date ON call_options_data(quote_date);"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_call_expire_date ON call_options_data(expire_date);"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_call_strike ON call_options_data(strike);"))

    # Add Indexes for PUT table
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_put_symbol ON put_options_data(symbol);"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_put_quote_date ON put_options_data(quote_date);"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_put_expire_date ON put_options_data(expire_date);"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_put_strike ON put_options_data(strike);"))

logger.info("Tables and indexes created successfully.")

2026-05-14 23:49:40,983 - INFO - Creating call_options_data table...
2026-05-14 23:49:40,984 - INFO - Creating put_options_data table...
2026-05-14 23:49:40,988 - INFO - Tables and indexes created successfully.


#### 1.5) Write Data into Staging Tables

In [9]:
# write df to the SQL db, replace if it exists
call_options_data.to_sql('staging_call_options_data', con=engine, if_exists='replace', index=False)
put_options_data.to_sql('staging_put_options_data', con=engine, if_exists='replace', index=False)

# Log number of rows inserted into the staging table
logger.info(f"Inserted {len(call_options_data)} rows into staging_call_options_data.")
logger.info(f"Inserted {len(put_options_data)} rows into staging_put_options_data.")

2026-05-14 23:49:45,818 - INFO - Inserted 37927 rows into staging_call_options_data.
2026-05-14 23:49:45,820 - INFO - Inserted 37927 rows into staging_put_options_data.


#### 1.6) Data Cleaning & Insertion into main Database

In [12]:
with engine.connect() as conn:

    # Remove duplicates for new CALL data
    del_call = """
    DELETE FROM staging_call_options_data
    WHERE EXISTS (
        SELECT 1 FROM call_options_data
        WHERE call_options_data.symbol = staging_call_options_data.symbol
        AND call_options_data.quote_date = staging_call_options_data.quote_date
        AND call_options_data.expire_date = staging_call_options_data.expire_date
        AND call_options_data.strike = staging_call_options_data.strike
    );
    """
    res = conn.execute(text(del_call))
    logger.info(f"Removed {res.rowcount} duplicate CALL rows.")


    # Insert new CALL rows (exclude ID column)
    ins_call = """
    INSERT INTO call_options_data (
        quote_date, symbol, underlying_last, expire_date, expiry_type, dte,
        strike, strike_distance, strike_distance_pct,
        c_delta, c_gamma, c_vega, c_theta, c_rho,
        c_iv, c_volume, c_last, c_size, c_bid, c_ask, c_mid, c_spread
    )
    SELECT
        quote_date, symbol, underlying_last, expire_date, expiry_type, dte,
        strike, strike_distance, strike_distance_pct,
        c_delta, c_gamma, c_vega, c_theta, c_rho,
        c_iv, c_volume, c_last, c_size, c_bid, c_ask, c_mid, c_spread
    FROM staging_call_options_data;
    """
    res = conn.execute(text(ins_call))
    logger.info(f"Inserted {res.rowcount} new CALL rows.")

    conn.execute(text("DELETE FROM staging_call_options_data"))


    # Remove duplicates for new PUT data
    del_put = """
    DELETE FROM staging_put_options_data
    WHERE EXISTS (
        SELECT 1 FROM put_options_data
        WHERE put_options_data.symbol = staging_put_options_data.symbol
        AND put_options_data.quote_date = staging_put_options_data.quote_date
        AND put_options_data.expire_date = staging_put_options_data.expire_date
        AND put_options_data.strike = staging_put_options_data.strike
    );
    """
    res = conn.execute(text(del_put))
    logger.info(f"Removed {res.rowcount} duplicate PUT rows.")


    # Insert new PUT rows (exclude ID column)
    ins_put = """
    INSERT INTO put_options_data (
        quote_date, symbol, underlying_last, expire_date, expiry_type, dte,
        strike, strike_distance, strike_distance_pct,
        p_delta, p_gamma, p_vega, p_theta, p_rho,
        p_iv, p_volume, p_last, p_size, p_bid, p_ask, p_mid, p_spread
    )
    SELECT
        quote_date, symbol, underlying_last, expire_date, expiry_type, dte,
        strike, strike_distance, strike_distance_pct,
        p_delta, p_gamma, p_vega, p_theta, p_rho,
        p_iv, p_volume, p_last, p_size, p_bid, p_ask, p_mid, p_spread
    FROM staging_put_options_data;
    """
    res = conn.execute(text(ins_put))
    logger.info(f"Inserted {res.rowcount} new PUT rows.")

    conn.execute(text("DELETE FROM staging_put_options_data"))

logger.info("Ingestion completed successfully.")

2026-05-14 23:51:37,553 - INFO - Removed 0 duplicate CALL rows.
2026-05-14 23:51:37,750 - INFO - Inserted 37927 new CALL rows.
2026-05-14 23:51:37,941 - INFO - Removed 0 duplicate PUT rows.
2026-05-14 23:51:38,200 - INFO - Inserted 37927 new PUT rows.
2026-05-14 23:51:38,212 - INFO - Ingestion completed successfully.


### 2) Production ETL Pipeline

- This is where you take your findings from the **Design & Testing Schema** Phase and Implement it for deployment

#### 2.1) Create Table Function for first time data ingestion

In [18]:
def create_table(option_type, option_type_prefix):
    """Creates a table for call or put options if it does not exist, and adds indexes."""

    table_name = f"{option_type}_options_data"

    create_table_query = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        quote_date TEXT NOT NULL,
        symbol TEXT NOT NULL,
        underlying_last REAL NOT NULL,
        expire_date TEXT NOT NULL,
        expiry_type TEXT,
        dte REAL,
        strike REAL NOT NULL,
        strike_distance REAL,
        strike_distance_pct REAL,
        {option_type_prefix}_delta REAL,
        {option_type_prefix}_gamma REAL,
        {option_type_prefix}_vega REAL,
        {option_type_prefix}_theta REAL,
        {option_type_prefix}_rho REAL,
        {option_type_prefix}_iv REAL,
        {option_type_prefix}_volume REAL,
        {option_type_prefix}_last REAL,
        {option_type_prefix}_size TEXT,
        {option_type_prefix}_bid REAL,
        {option_type_prefix}_ask REAL,
        {option_type_prefix}_mid REAL,
        {option_type_prefix}_spread REAL
    );
    """

    index_queries = [
        f"CREATE INDEX IF NOT EXISTS idx_{option_type}_symbol ON {table_name}(symbol);",
        f"CREATE INDEX IF NOT EXISTS idx_{option_type}_quote_date ON {table_name}(quote_date);",
        f"CREATE INDEX IF NOT EXISTS idx_{option_type}_expire_date ON {table_name}(expire_date);",
        f"CREATE INDEX IF NOT EXISTS idx_{option_type}_strike ON {table_name}(strike);"
    ]

    with engine.connect() as conn:
        conn.execute(text(create_table_query))
        for q in index_queries:
            conn.execute(text(q))

    logger.info(f"Table '{table_name}' created or verified with indexes.")


#### 2.2) Create Data Processing Function

In [19]:
def process_option_data(option_type, option_type_prefix, df):
    """Handles data ingestion, deduplication, and final insertion into the database."""

    # Ensure main table + indexes exist
    create_table(option_type, option_type_prefix)

    # Write incoming data to staging table
    staging_table = f"staging_{option_type}_options_data"
    df.to_sql(staging_table, con=engine, if_exists="replace", index=False)
    logger.info(f"Inserted {len(df)} rows into {staging_table}.")

    # 3. Deduplication + Insert
    with engine.begin() as conn:

        # Remove duplicates already in main table
        del_query = f"""
        DELETE FROM {staging_table}
        WHERE EXISTS (
            SELECT 1 FROM {option_type}_options_data
            WHERE {option_type}_options_data.symbol = {staging_table}.symbol
            AND {option_type}_options_data.quote_date = {staging_table}.quote_date
            AND {option_type}_options_data.expire_date = {staging_table}.expire_date
            AND {option_type}_options_data.strike = {staging_table}.strike
        );
        """
        res = conn.execute(text(del_query))
        logger.info(f"Removed {res.rowcount} duplicate rows from {staging_table}.")

        # Insert NEW rows (exclude autoincrement ID)
        insert_query = f"""
        INSERT INTO {option_type}_options_data (
            quote_date, symbol, underlying_last, expire_date, expiry_type, dte,
            strike, strike_distance, strike_distance_pct,
            {option_type_prefix}_delta, {option_type_prefix}_gamma, {option_type_prefix}_vega,
            {option_type_prefix}_theta, {option_type_prefix}_rho,
            {option_type_prefix}_iv, {option_type_prefix}_volume, {option_type_prefix}_last,
            {option_type_prefix}_size, {option_type_prefix}_bid, {option_type_prefix}_ask,
            {option_type_prefix}_mid, {option_type_prefix}_spread
        )
        SELECT
            quote_date, symbol, underlying_last, expire_date, expiry_type, dte,
            strike, strike_distance, strike_distance_pct,
            {option_type_prefix}_delta, {option_type_prefix}_gamma, {option_type_prefix}_vega,
            {option_type_prefix}_theta, {option_type_prefix}_rho,
            {option_type_prefix}_iv, {option_type_prefix}_volume, {option_type_prefix}_last,
            {option_type_prefix}_size, {option_type_prefix}_bid, {option_type_prefix}_ask,
            {option_type_prefix}_mid, {option_type_prefix}_spread
        FROM {staging_table};
        """
        res = conn.execute(text(insert_query))
        logger.info(f"Inserted {res.rowcount} new rows into {option_type}_options_data.")


        # Clear staging table
        conn.execute(text(f"DELETE FROM {staging_table}"))
        logger.info(f"Cleared {staging_table} after processing.")

    logger.info(f"{option_type.upper()} options data processing completed successfully!")


## Running ETL Data Pipeline

In [20]:
# Process call option data from the call_options_data dataframe
process_option_data("call", 'c', call_options_data)

# Process put option data from the put_options_data dataframe
process_option_data("put", 'p', put_options_data)

2026-05-14 23:59:27,083 - INFO - Table 'call_options_data' created or verified with indexes.


2026-05-14 23:59:28,251 - INFO - Inserted 37927 rows into staging_call_options_data.
2026-05-14 23:59:28,270 - INFO - Removed 0 duplicate rows from staging_call_options_data.
2026-05-14 23:59:28,446 - INFO - Inserted 37927 new rows into call_options_data.
2026-05-14 23:59:28,455 - INFO - Cleared staging_call_options_data after processing.
2026-05-14 23:59:28,595 - INFO - CALL options data processing completed successfully!
2026-05-14 23:59:28,597 - INFO - Table 'put_options_data' created or verified with indexes.
2026-05-14 23:59:29,504 - INFO - Inserted 37927 rows into staging_put_options_data.
2026-05-14 23:59:29,526 - INFO - Removed 0 duplicate rows from staging_put_options_data.
2026-05-14 23:59:29,714 - INFO - Inserted 37927 new rows into put_options_data.
2026-05-14 23:59:29,726 - INFO - Cleared staging_put_options_data after processing.
2026-05-14 23:59:29,835 - INFO - PUT options data processing completed successfully!
